# Home Health DME × Snowflake — Hands-On Lab

Welcome to the **SnowDay**! This notebook builds a complete Snowflake analytics platform for Home Health's three use cases using Q1 2026 synthetic data loaded from CSV.

| Use Case | Source Data | Goal |
|----------|-------------|------|
| **Denials Reduction** | Clearinghouse 835s, EHR billing | Reduce 17% denial rate to <5%, recover denied revenue |
| **Sales Data Analysis** | CRM + CMS Medicare (Marketplace) | Territory optimization, market penetration, rep performance |
| **Call Center Consolidation** | Avaya, Five9, RingCentral CDRs | Unified metrics across 700+ locations, 3 phone systems |

### What You'll Build
1. `HOME_HEALTH_DEMO` database — 6 schemas + 6 HIPAA-compliant roles
2. **224K+ records** from Q1 2026 CSV files loaded via COPY INTO
3. **Dynamic Tables** — Bronze/Silver/Gold medallion pipeline, no orchestration required
4. **RBAC + Dynamic Data Masking** — PHI protection with role-based unmasking
5. **Semantic View + Cortex Analyst** — Natural language queries across all 3 use cases
6. **Cortex Search** — RAG over 10 operational policy documents
7. **Snowflake Intelligence Agent** — Combines structured data + document AI
8. **FHIR R4 Pipeline** — EHR data ingestion via VARIANT + LATERAL FLATTEN
9. **Streamlit Dashboard** — 5-tab KPI app with embedded AI chatbot

> All patient data is **100% synthetic**. No real PHI is used. Run cells in order.

## Section 1: Environment Setup

### How Snowflake Compute Works
A **Virtual Warehouse** is Snowflake's compute engine — an on-demand query processor that:
- **Auto-suspends** after 60 seconds idle (you pay $0 while idle)
- **Auto-resumes** instantly on the next query
- **Auto-scales** to multiple clusters for concurrent users without any configuration

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE OR REPLACE DATABASE HOME_HEALTH_DEMO
    COMMENT = 'Home Health DME Analytics Demo - Denials, Sales, Call Center';

USE DATABASE HOME_HEALTH_DEMO;

CREATE OR REPLACE SCHEMA RAW_DATA       COMMENT = 'Raw data from source systems';
CREATE OR REPLACE SCHEMA TRANSFORMED    COMMENT = 'Dynamic Tables - Bronze/Silver/Gold pipeline';
CREATE OR REPLACE SCHEMA ANALYTICS      COMMENT = 'KPI views, semantic models, AI services';
CREATE OR REPLACE SCHEMA DOCUMENTS      COMMENT = 'Policy documents for Cortex Search';
CREATE OR REPLACE SCHEMA FHIR_RAW       COMMENT = 'Raw FHIR R4 bundles as VARIANT';
CREATE OR REPLACE SCHEMA FHIR_ANALYTICS COMMENT = 'Flattened FHIR views for analytics';

CREATE OR REPLACE ROLE BILLING_ADMIN     COMMENT = 'Revenue cycle and denials team';
CREATE OR REPLACE ROLE SALES_MANAGER    COMMENT = 'Sales leadership and territory management';
CREATE OR REPLACE ROLE CALL_CENTER_LEAD COMMENT = 'Call center operations';
CREATE OR REPLACE ROLE ANALYST          COMMENT = 'Cross-functional analytics';
CREATE OR REPLACE ROLE DATA_ENGINEER    COMMENT = 'Data pipeline management';
CREATE OR REPLACE ROLE EXECUTIVE        COMMENT = 'Executive KPI access';

GRANT ROLE BILLING_ADMIN     TO ROLE EXECUTIVE;
GRANT ROLE SALES_MANAGER    TO ROLE EXECUTIVE;
GRANT ROLE CALL_CENTER_LEAD TO ROLE EXECUTIVE;
GRANT ROLE ANALYST          TO ROLE EXECUTIVE;
GRANT ROLE EXECUTIVE        TO ROLE SYSADMIN;
GRANT ROLE DATA_ENGINEER    TO ROLE SYSADMIN;

-- Per-workload warehouses: pay-per-query, auto-suspend when idle
CREATE OR REPLACE WAREHOUSE HOME_HEALTH_LOAD_WH
    WAREHOUSE_SIZE = 'MEDIUM' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE
    MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 3
    COMMENT = 'ETL: suspends after 60s idle. Cost: $0 when not loading.';

CREATE OR REPLACE WAREHOUSE HOME_HEALTH_ANALYTICS_WH
    WAREHOUSE_SIZE = 'LARGE' AUTO_SUSPEND = 300 AUTO_RESUME = TRUE
    MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 5
    COMMENT = 'Analytics: auto-scales 1-5 clusters for concurrent dashboard users.';

CREATE OR REPLACE WAREHOUSE HOME_HEALTH_ADHOC_WH
    WAREHOUSE_SIZE = 'SMALL' AUTO_SUSPEND = 60 AUTO_RESUME = TRUE
    COMMENT = 'Ad-hoc: economy mode for exploration.';

GRANT USAGE ON WAREHOUSE HOME_HEALTH_LOAD_WH      TO ROLE DATA_ENGINEER;
GRANT USAGE ON WAREHOUSE HOME_HEALTH_ANALYTICS_WH TO ROLE BILLING_ADMIN;
GRANT USAGE ON WAREHOUSE HOME_HEALTH_ANALYTICS_WH TO ROLE SALES_MANAGER;
GRANT USAGE ON WAREHOUSE HOME_HEALTH_ANALYTICS_WH TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON WAREHOUSE HOME_HEALTH_ANALYTICS_WH TO ROLE ANALYST;
GRANT USAGE ON WAREHOUSE HOME_HEALTH_ANALYTICS_WH TO ROLE EXECUTIVE;

USE ROLE DATA_ENGINEER;
USE WAREHOUSE HOME_HEALTH_LOAD_WH;

SELECT 'Environment ready' AS status, CURRENT_DATABASE() AS db, CURRENT_WAREHOUSE() AS wh;

## Section 2: Stages and Data Upload

### Pre-Generated Data
All data files are included in the GitHub repo (`data/` and `documents/` folders). Upload them to Snowflake before running Section 3.

**Upload via SnowSQL (from repo root):**
```
PUT file://./data/locations.csv              @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/claims_submissions.csv     @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/claims_denials.csv         @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/denial_appeals.csv         @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/sales_rep_activity.csv     @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/physician_referrals.csv    @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/cms_respiratory_claims.csv @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/call_detail_records.csv    @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/call_agent_performance.csv @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/patient_satisfaction.csv   @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./data/home_health_fhir_bundle.json @HOME_HEALTH_DATA_STAGE AUTO_COMPRESS=FALSE;
PUT file://./documents/*.md @HOME_HEALTH_DOCUMENTS_STAGE AUTO_COMPRESS=FALSE;
```

> **Tip:** In Snowsight you can also drag-and-drop files to a stage via Data > Databases > stage name > Upload.

In [ ]:
USE ROLE DATA_ENGINEER;
USE DATABASE HOME_HEALTH_DEMO;
USE SCHEMA RAW_DATA;
USE WAREHOUSE HOME_HEALTH_LOAD_WH;

CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    SKIP_HEADER = 1
    NULL_IF = ('', 'NULL', 'None')
    EMPTY_FIELD_AS_NULL = TRUE;

CREATE OR REPLACE FILE FORMAT JSON_FORMAT
    TYPE = 'JSON'
    STRIP_OUTER_ARRAY = FALSE;

CREATE OR REPLACE STAGE HOME_HEALTH_DATA_STAGE
    FILE_FORMAT = CSV_FORMAT
    COMMENT = 'All CSV data files and FHIR JSON';

CREATE OR REPLACE STAGE HOME_HEALTH_DOCUMENTS_STAGE
    COMMENT = 'Policy documents for Cortex Search';

-- Verify files are staged after uploading
LIST @HOME_HEALTH_DATA_STAGE;

## Section 3: Load Data — COPY INTO (224K+ Records)

### Q1 2026 Dataset

| Table | Records | Description |
|-------|---------|-------------|
| LOCATIONS | 700 | Operating centers across 48 states |
| CLAIMS_SUBMISSIONS | 50,000 | All DME claims submitted Q1 2026 |
| CLAIMS_DENIALS | ~8,459 | Denied claims with CARC/RARC codes |
| DENIAL_APPEALS | ~4,643 | Appeal outcomes and recovered revenue |
| SALES_REP_ACTIVITY | 15,000 | CRM activities across 150 reps |
| PHYSICIAN_REFERRALS | 12,000 | Physician equipment referrals |
| CMS_RESPIRATORY_CLAIMS | 30,000 | Medicare market data (Q4 2025) |
| CALL_DETAIL_RECORDS | 100,000 | CDRs from Avaya, Five9, RingCentral |
| CALL_AGENT_PERFORMANCE | 500 | Agent performance baselines |
| PATIENT_SATISFACTION | 8,000 | Post-call surveys |

`COPY INTO` handles parallel loading, type coercion, and error isolation automatically. `FORCE = TRUE` bypasses Snowflake's load history tracking — useful when re-running the HOL.

In [ ]:
USE SCHEMA HOME_HEALTH_DEMO.RAW_DATA;

CREATE OR REPLACE TABLE LOCATIONS (
    location_id VARCHAR(10), location_name VARCHAR(200), address VARCHAR(200),
    city VARCHAR(100), state VARCHAR(2), zip_code VARCHAR(10), region VARCHAR(50),
    phone VARCHAR(20), manager VARCHAR(100), open_date DATE, is_active BOOLEAN,
    square_footage INT, employee_count INT);

CREATE OR REPLACE TABLE PAYER_CONTRACTS (
    contract_id VARCHAR(10), payer_code VARCHAR(20), payer_name VARCHAR(100),
    plan_type VARCHAR(50), effective_date DATE, termination_date DATE,
    reimbursement_rate_pct FLOAT, timely_filing_days INT,
    prior_auth_required BOOLEAN, cmn_required BOOLEAN,
    electronic_submission BOOLEAN, contact_phone VARCHAR(20));

CREATE OR REPLACE TABLE CLAIMS_SUBMISSIONS (
    claim_id VARCHAR(15), patient_id VARCHAR(15), location_id VARCHAR(10),
    payer_code VARCHAR(20), payer_name VARCHAR(100), hcpcs_code VARCHAR(10),
    equipment_name VARCHAR(100), equipment_category VARCHAR(50), quantity INT,
    billed_amount FLOAT, allowed_amount FLOAT, submission_date DATE, service_date DATE,
    referring_physician_npi VARCHAR(15), referring_physician_name VARCHAR(100),
    physician_specialty VARCHAR(50), diagnosis_code VARCHAR(10), modifier VARCHAR(5),
    place_of_service VARCHAR(5), claim_status VARCHAR(20), adjudication_date DATE,
    paid_amount FLOAT, has_cmn BOOLEAN, prior_auth_obtained BOOLEAN);

CREATE OR REPLACE TABLE CLAIMS_DENIALS (
    denial_id VARCHAR(12), claim_id VARCHAR(15), patient_id VARCHAR(15),
    location_id VARCHAR(10), payer_code VARCHAR(20), payer_name VARCHAR(100),
    hcpcs_code VARCHAR(10), equipment_category VARCHAR(50), denial_date DATE,
    denial_code VARCHAR(10), denial_reason VARCHAR(200), denial_category VARCHAR(50),
    root_cause VARCHAR(50), billed_amount FLOAT, denied_amount FLOAT,
    assigned_to VARCHAR(100), priority VARCHAR(10), status VARCHAR(20),
    days_to_resolve INT, is_repeat_denial BOOLEAN, original_claim_clean BOOLEAN);

CREATE OR REPLACE TABLE DENIAL_APPEALS (
    appeal_id VARCHAR(12), denial_id VARCHAR(12), claim_id VARCHAR(15),
    location_id VARCHAR(10), payer_code VARCHAR(20), appeal_date DATE,
    appeal_level VARCHAR(30), appeal_reason VARCHAR(200), supporting_docs VARCHAR(100),
    outcome VARCHAR(20), outcome_date DATE, recovered_amount FLOAT,
    appeal_specialist VARCHAR(100), turnaround_days INT, payer_response_notes VARCHAR(500));

CREATE OR REPLACE TABLE SALES_REP_ACTIVITY (
    activity_id VARCHAR(12), rep_id VARCHAR(10), rep_name VARCHAR(100),
    territory VARCHAR(50), activity_date DATE, activity_type VARCHAR(30),
    physician_npi VARCHAR(15), physician_name VARCHAR(100), physician_specialty VARCHAR(50),
    facility_name VARCHAR(200), location_id VARCHAR(10), state VARCHAR(2),
    outcome VARCHAR(30), referral_generated BOOLEAN, notes VARCHAR(500),
    duration_minutes INT, miles_driven FLOAT);

CREATE OR REPLACE TABLE PHYSICIAN_REFERRALS (
    referral_id VARCHAR(12), physician_npi VARCHAR(15), physician_name VARCHAR(100),
    physician_specialty VARCHAR(50), facility_name VARCHAR(200), patient_id VARCHAR(15),
    referral_date DATE, equipment_category VARCHAR(50), hcpcs_code VARCHAR(10),
    equipment_name VARCHAR(100), location_id VARCHAR(10), state VARCHAR(2),
    referral_source VARCHAR(50), status VARCHAR(20), days_to_setup INT,
    revenue FLOAT, is_new_physician BOOLEAN);

CREATE OR REPLACE TABLE CMS_RESPIRATORY_CLAIMS (
    cms_claim_id VARCHAR(15), reporting_quarter VARCHAR(10), state VARCHAR(2),
    city VARCHAR(100), zip_code VARCHAR(10), hcpcs_code VARCHAR(10),
    equipment_category VARCHAR(50), beneficiary_count INT, total_claims INT,
    total_allowed FLOAT, total_paid FLOAT, provider_type VARCHAR(50),
    home_health_share BOOLEAN, competitor_count INT);

CREATE OR REPLACE TABLE CALL_DETAIL_RECORDS (
    cdr_id VARCHAR(15), phone_system VARCHAR(20), call_date DATE, call_time TIME,
    direction VARCHAR(10), call_type VARCHAR(30), caller_id VARCHAR(20),
    patient_id VARCHAR(15), agent_id VARCHAR(10), agent_name VARCHAR(100),
    location_id VARCHAR(10), queue_name VARCHAR(30), wait_time_seconds INT,
    handle_time_seconds INT, after_call_work_seconds INT, total_duration_seconds INT,
    abandoned BOOLEAN, transferred BOOLEAN, first_call_resolution BOOLEAN,
    disposition VARCHAR(30), satisfaction_score INT, recording_available BOOLEAN);

CREATE OR REPLACE TABLE CALL_AGENT_PERFORMANCE (
    agent_id VARCHAR(10), agent_name VARCHAR(100), location_id VARCHAR(10),
    phone_system VARCHAR(20), team VARCHAR(30), hire_date DATE,
    avg_handle_time_seconds INT, avg_after_call_work_seconds INT,
    first_call_resolution_rate FLOAT, calls_per_hour FLOAT,
    avg_satisfaction_score FLOAT, adherence_rate FLOAT, utilization_rate FLOAT,
    quality_score FLOAT, escalation_rate FLOAT, is_active BOOLEAN);

CREATE OR REPLACE TABLE PATIENT_SATISFACTION (
    survey_id VARCHAR(12), patient_id VARCHAR(15), survey_date DATE,
    channel VARCHAR(10), interaction_type VARCHAR(30), overall_score INT,
    ease_of_contact INT, agent_knowledge INT, issue_resolved BOOLEAN,
    wait_time_acceptable BOOLEAN, would_recommend BOOLEAN, comments VARCHAR(500),
    location_id VARCHAR(10));

SELECT 'All tables created' AS status;

In [ ]:
COPY INTO LOCATIONS              FROM @HOME_HEALTH_DATA_STAGE/locations.csv              FILE_FORMAT = CSV_FORMAT;
COPY INTO PAYER_CONTRACTS         FROM @HOME_HEALTH_DATA_STAGE/payer_contracts.csv         FILE_FORMAT = CSV_FORMAT;
COPY INTO CLAIMS_SUBMISSIONS      FROM @HOME_HEALTH_DATA_STAGE/claims_submissions.csv      FILE_FORMAT = CSV_FORMAT;
COPY INTO CLAIMS_DENIALS          FROM @HOME_HEALTH_DATA_STAGE/claims_denials.csv          FILE_FORMAT = CSV_FORMAT;
COPY INTO DENIAL_APPEALS          FROM @HOME_HEALTH_DATA_STAGE/denial_appeals.csv          FILE_FORMAT = CSV_FORMAT;
COPY INTO SALES_REP_ACTIVITY      FROM @HOME_HEALTH_DATA_STAGE/sales_rep_activity.csv      FILE_FORMAT = CSV_FORMAT;
COPY INTO PHYSICIAN_REFERRALS     FROM @HOME_HEALTH_DATA_STAGE/physician_referrals.csv     FILE_FORMAT = CSV_FORMAT;
COPY INTO CMS_RESPIRATORY_CLAIMS  FROM @HOME_HEALTH_DATA_STAGE/cms_respiratory_claims.csv  FILE_FORMAT = CSV_FORMAT;
COPY INTO CALL_DETAIL_RECORDS     FROM @HOME_HEALTH_DATA_STAGE/call_detail_records.csv     FILE_FORMAT = CSV_FORMAT;
COPY INTO CALL_AGENT_PERFORMANCE  FROM @HOME_HEALTH_DATA_STAGE/call_agent_performance.csv  FILE_FORMAT = CSV_FORMAT;
COPY INTO PATIENT_SATISFACTION    FROM @HOME_HEALTH_DATA_STAGE/patient_satisfaction.csv    FILE_FORMAT = CSV_FORMAT;

SELECT 'CLAIMS_SUBMISSIONS'    AS tbl, COUNT(*) AS rows FROM CLAIMS_SUBMISSIONS
UNION ALL SELECT 'CLAIMS_DENIALS',      COUNT(*) FROM CLAIMS_DENIALS
UNION ALL SELECT 'CALL_DETAIL_RECORDS', COUNT(*) FROM CALL_DETAIL_RECORDS
UNION ALL SELECT 'SALES_REP_ACTIVITY',  COUNT(*) FROM SALES_REP_ACTIVITY
UNION ALL SELECT 'PHYSICIAN_REFERRALS', COUNT(*) FROM PHYSICIAN_REFERRALS
UNION ALL SELECT 'LOCATIONS',           COUNT(*) FROM LOCATIONS
ORDER BY rows DESC;

## Section 4: Data Governance — PHI/PII Classification and Masking

### Why Governance Matters for HIPAA-Regulated DME Operations
Before Snowflake, controlling PHI access required: separate de-identified data copies (expensive and stale), manual column redaction scripts (error-prone), and multiple layered database permissions (complex to audit).

**Snowflake Horizon** solves this with one policy engine across all tables:

| Capability | What It Does |
|------------|--------------|
| **Dynamic Masking** | PHI masked at query time based on role — zero data copies |
| **Data Classification Tags** | PHI/PII/FINANCIAL labels for audit and compliance |
| **Row Access Policies** | Location-based filtering — billing in FL sees only FL claims |
| **Time Travel** | 90-day audit trail — required for CMS Medicare compliance |

### The Magic: One Table, Two Realities
```
ANALYST role:       patient_id = 'PAT-***MASKED***'
BILLING_ADMIN role: patient_id = 'PAT-0015473'
```
Same table. Same SQL query. Different result based on your role. No copies. No pipeline.

> **Important:** Run `USE SECONDARY ROLES NONE` before testing masking. By default, secondary roles are enabled — ACCOUNTADMIN privileges will override masking policies without this step.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE SCHEMA RAW_DATA;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

-- Unset any existing masking policies before recreating
-- (Cannot CREATE OR REPLACE a policy that is already applied to a column)
ALTER TABLE IF EXISTS RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN patient_id              UNSET MASKING POLICY;
ALTER TABLE IF EXISTS RAW_DATA.CLAIMS_DENIALS      MODIFY COLUMN patient_id              UNSET MASKING POLICY;
ALTER TABLE IF EXISTS RAW_DATA.CALL_DETAIL_RECORDS MODIFY COLUMN caller_id              UNSET MASKING POLICY;
ALTER TABLE IF EXISTS RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN referring_physician_npi UNSET MASKING POLICY;

-- Patient ID masking (PHI - HIPAA required)
CREATE OR REPLACE MASKING POLICY RAW_DATA.MASK_PATIENT_ID AS (val VARCHAR)
RETURNS VARCHAR ->
    CASE WHEN CURRENT_ROLE() IN ('DATA_ENGINEER','BILLING_ADMIN','EXECUTIVE') THEN val
         ELSE 'PAT-***MASKED***' END;

-- Phone/caller ID masking (PII)
CREATE OR REPLACE MASKING POLICY RAW_DATA.MASK_PHONE AS (val VARCHAR)
RETURNS VARCHAR ->
    CASE WHEN CURRENT_ROLE() IN ('DATA_ENGINEER','CALL_CENTER_LEAD','EXECUTIVE') THEN val
         ELSE REGEXP_REPLACE(val, '[0-9]', '*') END;

-- NPI masking
CREATE OR REPLACE MASKING POLICY RAW_DATA.MASK_NPI AS (val VARCHAR)
RETURNS VARCHAR ->
    CASE WHEN CURRENT_ROLE() IN ('DATA_ENGINEER','SALES_MANAGER','BILLING_ADMIN','EXECUTIVE') THEN val
         ELSE LEFT(val, 3) || '****' || RIGHT(val, 3) END;

-- Apply masking policies to PHI columns
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN patient_id              SET MASKING POLICY RAW_DATA.MASK_PATIENT_ID;
ALTER TABLE RAW_DATA.CLAIMS_DENIALS      MODIFY COLUMN patient_id              SET MASKING POLICY RAW_DATA.MASK_PATIENT_ID;
ALTER TABLE RAW_DATA.CALL_DETAIL_RECORDS MODIFY COLUMN caller_id              SET MASKING POLICY RAW_DATA.MASK_PHONE;
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN referring_physician_npi SET MASKING POLICY RAW_DATA.MASK_NPI;

-- Data classification tags for audit and compliance reporting
CREATE OR REPLACE TAG RAW_DATA.DATA_SENSITIVITY ALLOWED_VALUES 'PHI','PII','FINANCIAL','PUBLIC';
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN patient_id     SET TAG RAW_DATA.DATA_SENSITIVITY = 'PHI';
ALTER TABLE RAW_DATA.CALL_DETAIL_RECORDS MODIFY COLUMN caller_id     SET TAG RAW_DATA.DATA_SENSITIVITY = 'PII';
ALTER TABLE RAW_DATA.CLAIMS_SUBMISSIONS  MODIFY COLUMN billed_amount  SET TAG RAW_DATA.DATA_SENSITIVITY = 'FINANCIAL';

-- Grant table access to roles
GRANT SELECT ON ALL TABLES IN SCHEMA RAW_DATA TO ROLE BILLING_ADMIN;
GRANT SELECT ON ALL TABLES IN SCHEMA RAW_DATA TO ROLE ANALYST;
GRANT SELECT ON TABLE RAW_DATA.SALES_REP_ACTIVITY     TO ROLE SALES_MANAGER;
GRANT SELECT ON TABLE RAW_DATA.PHYSICIAN_REFERRALS    TO ROLE SALES_MANAGER;
GRANT SELECT ON TABLE RAW_DATA.CMS_RESPIRATORY_CLAIMS TO ROLE SALES_MANAGER;
GRANT SELECT ON TABLE RAW_DATA.CALL_DETAIL_RECORDS    TO ROLE CALL_CENTER_LEAD;
GRANT SELECT ON TABLE RAW_DATA.CALL_AGENT_PERFORMANCE TO ROLE CALL_CENTER_LEAD;
GRANT SELECT ON TABLE RAW_DATA.PATIENT_SATISFACTION   TO ROLE CALL_CENTER_LEAD;

-- ── TEST: disable secondary roles first — ACCOUNTADMIN overrides masking otherwise
USE SECONDARY ROLES NONE;

USE ROLE ANALYST;
SELECT claim_id, patient_id, billed_amount
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS LIMIT 3;
-- patient_id SHOULD show as 'PAT-***MASKED***'

USE ROLE BILLING_ADMIN;
SELECT claim_id, patient_id, billed_amount
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS LIMIT 3;
-- patient_id SHOULD show real value e.g. 'PAT-0015473'

-- Always reset secondary roles after testing
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES ALL;

## Section 5: Dynamic Tables — Declarative Data Pipeline

### No Orchestration Required
Traditional pipelines need Airflow, Data Factory, or hand-written DAGs. With Dynamic Tables, you declare **what** you want and Snowflake automatically figures out **when** and **how** to refresh.

```
RAW DATA
    ↓
BRONZE (1-min lag)  — Cleanse and validate: trim whitespace, coalesce nulls, classify clean/denied
    ↓
SILVER (5-min lag)  — Aggregate KPIs: denial rates by payer, call SLAs by phone system, weekly trends
    ↓
GOLD (DOWNSTREAM)   — Executive KPIs: refreshes automatically when Silver refreshes
    ↓
ALERTS (1-min lag)  — Real-time denial spikes and call SLA breaches
```

Each layer only refreshes when source data changes — incremental compute, not full reprocessing.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

-- ── BRONZE LAYER: Cleanse raw data (1-minute lag) ─────────────────────────

CREATE OR REPLACE DYNAMIC TABLE TRANSFORMED.DT_CLEAN_CLAIMS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT claim_id, patient_id, location_id, UPPER(TRIM(payer_code)) AS payer_code,
       payer_name, UPPER(TRIM(hcpcs_code)) AS hcpcs_code, equipment_category,
       quantity, COALESCE(billed_amount,0) AS billed_amount, COALESCE(paid_amount,0) AS paid_amount,
       submission_date, adjudication_date,
       DATEDIFF(day, submission_date, COALESCE(adjudication_date, CURRENT_DATE())) AS days_in_ar,
       CASE WHEN claim_status='PAID' AND paid_amount>0 THEN TRUE ELSE FALSE END AS is_clean_claim,
       CASE WHEN claim_status='DENIED' THEN TRUE ELSE FALSE END AS is_denied,
       UPPER(TRIM(claim_status)) AS claim_status
FROM RAW_DATA.CLAIMS_SUBMISSIONS WHERE claim_id IS NOT NULL;

CREATE OR REPLACE DYNAMIC TABLE TRANSFORMED.DT_CLEAN_DENIALS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT denial_id, claim_id, patient_id, location_id,
       UPPER(TRIM(payer_code)) AS payer_code, payer_name,
       UPPER(TRIM(hcpcs_code)) AS hcpcs_code, equipment_category,
       denial_date, denial_code, denial_reason,
       UPPER(TRIM(denial_category)) AS denial_category, UPPER(TRIM(root_cause)) AS root_cause,
       COALESCE(denied_amount,0) AS denied_amount, UPPER(TRIM(priority)) AS priority,
       UPPER(TRIM(status)) AS status, days_to_resolve, is_repeat_denial
FROM RAW_DATA.CLAIMS_DENIALS WHERE denial_id IS NOT NULL;

CREATE OR REPLACE DYNAMIC TABLE TRANSFORMED.DT_CLEAN_CALL_RECORDS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT cdr_id, UPPER(TRIM(phone_system)) AS phone_system, call_date, call_time,
       UPPER(TRIM(direction)) AS direction, INITCAP(TRIM(call_type)) AS call_type,
       agent_id, agent_name, location_id, queue_name,
       COALESCE(wait_time_seconds,0) AS wait_time_seconds,
       COALESCE(handle_time_seconds,0) AS handle_time_seconds,
       abandoned, transferred, first_call_resolution, satisfaction_score,
       CASE WHEN wait_time_seconds<=20 THEN 'WITHIN_SLA'
            WHEN wait_time_seconds<=60 THEN 'NEAR_SLA' ELSE 'SLA_BREACH' END AS sla_status
FROM RAW_DATA.CALL_DETAIL_RECORDS WHERE cdr_id IS NOT NULL;

SELECT 'Bronze layer ready' AS status;

In [ ]:
-- ── SILVER LAYER: Business KPIs (5-minute lag) ────────────────────────────

CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_DENIAL_RATE_BY_PAYER
TARGET_LAG = '5 minutes' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT payer_code, payer_name, DATE_TRUNC('week', submission_date) AS week_start,
       COUNT(*) AS total_claims,
       SUM(CASE WHEN is_denied THEN 1 ELSE 0 END) AS denied_claims,
       ROUND(SUM(CASE WHEN is_denied THEN 1 ELSE 0 END)*100.0/NULLIF(COUNT(*),0),2) AS denial_rate_pct,
       ROUND(SUM(CASE WHEN is_clean_claim THEN 1 ELSE 0 END)*100.0/NULLIF(COUNT(*),0),2) AS clean_claim_rate_pct,
       SUM(billed_amount) AS total_billed, SUM(paid_amount) AS total_paid,
       ROUND(AVG(days_in_ar),1) AS avg_days_in_ar
FROM TRANSFORMED.DT_CLEAN_CLAIMS
GROUP BY payer_code, payer_name, DATE_TRUNC('week', submission_date);

CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_CALL_CENTER_METRICS
TARGET_LAG = '5 minutes' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT phone_system, location_id, call_type, DATE_TRUNC('day', call_date) AS call_day,
       COUNT(*) AS total_calls,
       COUNT(CASE WHEN abandoned THEN 1 END) AS abandoned_count,
       ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/NULLIF(COUNT(*),0),2) AS abandonment_rate_pct,
       ROUND(AVG(wait_time_seconds),1) AS avg_wait_seconds,
       ROUND(AVG(handle_time_seconds),1) AS avg_handle_seconds,
       ROUND(COUNT(CASE WHEN first_call_resolution THEN 1 END)*100.0/NULLIF(COUNT(CASE WHEN NOT abandoned THEN 1 END),0),2) AS fcr_rate_pct,
       ROUND(COUNT(CASE WHEN sla_status='WITHIN_SLA' THEN 1 END)*100.0/NULLIF(COUNT(CASE WHEN direction='INBOUND' THEN 1 END),0),2) AS service_level_pct
FROM TRANSFORMED.DT_CLEAN_CALL_RECORDS
GROUP BY phone_system, location_id, call_type, DATE_TRUNC('day', call_date);

-- ── GOLD LAYER (DOWNSTREAM) — refreshes automatically when Silver refreshes ──
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_EXECUTIVE_KPIS
TARGET_LAG = DOWNSTREAM WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT 'Q1 2026' AS reporting_period, CURRENT_TIMESTAMP() AS last_refresh,
    (SELECT ROUND(AVG(denial_rate_pct),2) FROM ANALYTICS.DT_DENIAL_RATE_BY_PAYER) AS overall_denial_rate_pct,
    (SELECT ROUND(AVG(clean_claim_rate_pct),2) FROM ANALYTICS.DT_DENIAL_RATE_BY_PAYER) AS clean_claim_rate_pct,
    (SELECT ROUND(AVG(avg_days_in_ar),1) FROM ANALYTICS.DT_DENIAL_RATE_BY_PAYER) AS avg_days_in_ar,
    (SELECT SUM(total_calls) FROM ANALYTICS.DT_CALL_CENTER_METRICS) AS total_calls_q1,
    (SELECT ROUND(AVG(abandonment_rate_pct),2) FROM ANALYTICS.DT_CALL_CENTER_METRICS) AS avg_abandonment_pct,
    (SELECT ROUND(AVG(fcr_rate_pct),2) FROM ANALYTICS.DT_CALL_CENTER_METRICS) AS avg_fcr_rate_pct;

-- ── ALERTS: Real-time denial spikes (1-minute lag) ────────────────────────
CREATE OR REPLACE DYNAMIC TABLE ANALYTICS.DT_REALTIME_ALERTS
TARGET_LAG = '1 minute' WAREHOUSE = HOME_HEALTH_ANALYTICS_WH AS
SELECT 'DENIAL_SPIKE' AS alert_type, 'HIGH' AS severity, location_id,
       'Denial spike: '||COUNT(*)||' denials in last 7 days' AS alert_message,
       COUNT(*) AS metric_value, CURRENT_TIMESTAMP() AS alert_timestamp
FROM TRANSFORMED.DT_CLEAN_DENIALS
WHERE denial_date >= DATEADD(day,-7,CURRENT_DATE())
GROUP BY location_id HAVING COUNT(*) > 15
UNION ALL
SELECT 'CALL_ABANDONMENT','CRITICAL', location_id,
       'High abandonment: '||ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/COUNT(*),1)||'% on '||phone_system,
       ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/NULLIF(COUNT(*),0),1), CURRENT_TIMESTAMP()
FROM TRANSFORMED.DT_CLEAN_CALL_RECORDS WHERE call_date >= DATEADD(day,-1,CURRENT_DATE())
GROUP BY location_id, phone_system
HAVING ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/NULLIF(COUNT(*),0),1) > 15;

GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA HOME_HEALTH_DEMO.ANALYTICS   TO ROLE BILLING_ADMIN;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA HOME_HEALTH_DEMO.ANALYTICS   TO ROLE SALES_MANAGER;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA HOME_HEALTH_DEMO.ANALYTICS   TO ROLE CALL_CENTER_LEAD;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA HOME_HEALTH_DEMO.ANALYTICS   TO ROLE ANALYST;
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA HOME_HEALTH_DEMO.ANALYTICS   TO ROLE EXECUTIVE;

SHOW DYNAMIC TABLES IN DATABASE HOME_HEALTH_DEMO;

## Section 6: Domain Analytics — Three Use Case Deep Dives

### Three Use Cases, One Platform
Home Health data has historically lived in silos: billing systems (clearinghouse), CRM, and three separate phone systems. The queries below are only possible because everything is in one place.
- **Use Case 1 — Denials:** Root cause Pareto, payer-level trending, appeal recovery funnel
- **Use Case 2 — Sales:** Internal CRM against CMS Medicare market data to show true market share
- **Use Case 3 — Call Center:** Avaya, Five9, RingCentral in one `GROUP BY` — previously required months of ETL work

The cross-domain insight at the end — denied claims correlating to billing inquiry call spikes — is only visible when all data is on one unified platform.

In [ ]:
-- ── USE CASE 1: DENIALS — Root cause analysis ─────────────────────────────

-- Denial rate by payer (identify worst payers first)
SELECT payer_name, COUNT(*) AS total_claims,
       COUNT(CASE WHEN claim_status='DENIED' THEN 1 END) AS denied,
       ROUND(COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)*100.0/COUNT(*),2) AS denial_rate_pct,
       ROUND(COUNT(CASE WHEN claim_status='PAID' AND paid_amount>0 THEN 1 END)*100.0/COUNT(*),2) AS clean_claim_rate_pct
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS
GROUP BY payer_name ORDER BY denial_rate_pct DESC;

-- Root cause Pareto (top 3 causes typically = 80% of denials)
SELECT root_cause, denial_code, denial_reason,
       COUNT(*) AS denial_count, SUM(denied_amount) AS total_denied_revenue,
       ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER (),1) AS pct_of_total
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS
GROUP BY root_cause, denial_code, denial_reason
ORDER BY denial_count DESC LIMIT 10;

-- Recovery funnel: denied → appealed → recovered
SELECT 'Total Claims' AS stage, COUNT(*) AS count, SUM(billed_amount) AS amount
FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS
UNION ALL SELECT 'Denied',     COUNT(*), SUM(denied_amount)  FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS
UNION ALL SELECT 'Appealed',   COUNT(*), SUM(d.denied_amount)
FROM HOME_HEALTH_DEMO.RAW_DATA.DENIAL_APPEALS a
JOIN HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS d ON a.denial_id=d.denial_id
UNION ALL SELECT 'Recovered',  COUNT(*), SUM(recovered_amount)
FROM HOME_HEALTH_DEMO.RAW_DATA.DENIAL_APPEALS WHERE outcome IN ('OVERTURNED','PARTIAL');

In [ ]:
-- ── USE CASE 2: SALES — Internal CRM vs CMS Medicare market data ──────────
-- In production, CMS data comes from Snowflake Marketplace (Cybersyn, Definitive Healthcare)
-- Zero ETL: click Get in the Marketplace and query it directly alongside your own data
SELECT cms.state,
       SUM(cms.total_claims) AS total_market_claims,
       SUM(CASE WHEN cms.home_health_share THEN cms.total_claims ELSE 0 END) AS our_claims,
       ROUND(SUM(CASE WHEN cms.home_health_share THEN cms.total_claims ELSE 0 END)*100.0/NULLIF(SUM(cms.total_claims),0),2) AS market_share_pct,
       COUNT(DISTINCT r.physician_npi) AS referring_physicians
FROM HOME_HEALTH_DEMO.RAW_DATA.CMS_RESPIRATORY_CLAIMS cms
LEFT JOIN HOME_HEALTH_DEMO.RAW_DATA.PHYSICIAN_REFERRALS r ON cms.state = r.state
GROUP BY cms.state ORDER BY market_share_pct DESC LIMIT 15;

-- ── USE CASE 3: CALL CENTER — Three phone systems, one query ──────────────
SELECT phone_system, COUNT(*) AS total_calls,
       ROUND(AVG(handle_time_seconds)/60.0,1) AS avg_aht_min,
       ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/NULLIF(COUNT(*),0),2) AS abandonment_pct,
       ROUND(COUNT(CASE WHEN first_call_resolution THEN 1 END)*100.0/NULLIF(COUNT(CASE WHEN NOT abandoned THEN 1 END),0),2) AS fcr_pct,
       COUNT(DISTINCT agent_id) AS agents
FROM HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS
GROUP BY phone_system ORDER BY total_calls DESC;

-- ── CROSS-DOMAIN: Denied claims → billing call spikes (only possible on unified platform)
SELECT l.location_id, l.state, l.region,
       COALESCE(d.denial_count,0) AS denial_count,
       COALESCE(c.billing_calls,0) AS billing_calls,
       CASE WHEN COALESCE(d.denial_count,0)>20 AND COALESCE(c.billing_calls,0)>50 THEN 'HIGH_RISK'
            WHEN COALESCE(d.denial_count,0)>10 OR COALESCE(c.billing_calls,0)>25 THEN 'MEDIUM_RISK'
            ELSE 'LOW_RISK' END AS risk_category
FROM HOME_HEALTH_DEMO.RAW_DATA.LOCATIONS l
LEFT JOIN (SELECT location_id, COUNT(*) AS denial_count FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS GROUP BY 1) d ON l.location_id=d.location_id
LEFT JOIN (SELECT location_id, COUNT(CASE WHEN call_type='Billing Inquiry' THEN 1 END) AS billing_calls FROM HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS GROUP BY 1) c ON l.location_id=c.location_id
WHERE l.is_active=TRUE ORDER BY denial_count DESC LIMIT 15;

## Section 7: Cortex Analyst — Natural Language Analytics

### What it Does
Cortex Analyst translates plain English questions into accurate SQL — returning answers without anyone needing to know SQL, know your data model, or wait for a report.

**Billing teams can ask:** "Which payer has the highest denial rate this month?"
**Sales managers can ask:** "Which territories have the lowest market penetration?"
**Call center leads can ask:** "How did our abandonment rate compare across phone systems last week?"

### Semantic Views — The Translation Layer
A **Semantic View** is a native SQL object that tells Cortex Analyst:
- What each table means (business descriptions and synonyms)
- How tables relate to each other (join paths)
- What metrics mean (`denial_rate` = denied claims / total claims × 100)
- Verified example questions with their correct SQL (AI_VERIFIED_QUERIES)

Once created, any natural language question across all three use cases gets accurately translated to SQL.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE SCHEMA ANALYTICS;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

CREATE OR REPLACE SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS
  TABLES (
    CLAIMS_SUBMISSIONS AS HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS
      PRIMARY KEY (claim_id) WITH SYNONYMS = ('claims','submissions','billing')
      COMMENT = 'DME claims submitted to payers Q1 2026',
    CLAIMS_DENIALS AS HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS
      PRIMARY KEY (denial_id) WITH SYNONYMS = ('denials','rejected claims','denied')
      COMMENT = 'Claims denied by payers with CARC codes',
    DENIAL_APPEALS AS HOME_HEALTH_DEMO.RAW_DATA.DENIAL_APPEALS
      PRIMARY KEY (appeal_id) WITH SYNONYMS = ('appeals','rework','overturned')
      COMMENT = 'Appeals filed against denied claims',
    LOCATIONS AS HOME_HEALTH_DEMO.RAW_DATA.LOCATIONS
      PRIMARY KEY (location_id) WITH SYNONYMS = ('centers','branches','offices','sites')
      COMMENT = 'Home Health operating centers across 48 states',
    SALES_REP_ACTIVITY AS HOME_HEALTH_DEMO.RAW_DATA.SALES_REP_ACTIVITY
      PRIMARY KEY (activity_id) WITH SYNONYMS = ('sales activities','rep visits','outreach')
      COMMENT = 'Sales rep activities and physician contacts',
    PHYSICIAN_REFERRALS AS HOME_HEALTH_DEMO.RAW_DATA.PHYSICIAN_REFERRALS
      PRIMARY KEY (referral_id) WITH SYNONYMS = ('referrals','physician orders')
      COMMENT = 'Physician referrals for DME equipment',
    CALL_DETAIL_RECORDS AS HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS
      PRIMARY KEY (cdr_id) WITH SYNONYMS = ('calls','CDR','phone calls','call records')
      COMMENT = 'Call records from Avaya, Five9, and RingCentral'
  )
  RELATIONSHIPS (
    denials_to_claims    AS CLAIMS_DENIALS (claim_id)        REFERENCES CLAIMS_SUBMISSIONS (claim_id),
    appeals_to_denials   AS DENIAL_APPEALS (denial_id)       REFERENCES CLAIMS_DENIALS (denial_id),
    claims_to_locations  AS CLAIMS_SUBMISSIONS (location_id) REFERENCES LOCATIONS (location_id),
    denials_to_locations AS CLAIMS_DENIALS (location_id)     REFERENCES LOCATIONS (location_id),
    sales_to_locations   AS SALES_REP_ACTIVITY (location_id) REFERENCES LOCATIONS (location_id),
    calls_to_locations   AS CALL_DETAIL_RECORDS (location_id) REFERENCES LOCATIONS (location_id)
  )
  DIMENSIONS (
    CLAIMS_SUBMISSIONS.payer_name_dim    AS payer_name       COMMENT = 'Payer Name',
    CLAIMS_SUBMISSIONS.claim_status_dim  AS claim_status     COMMENT = 'Claim Status',
    CLAIMS_SUBMISSIONS.submission_date_dim AS submission_date COMMENT = 'Submission Date',
    CLAIMS_DENIALS.denial_code_dim       AS denial_code      COMMENT = 'Denial Code (CARC)',
    CLAIMS_DENIALS.root_cause_dim        AS root_cause       COMMENT = 'Root Cause',
    LOCATIONS.state_dim                  AS state            COMMENT = 'State',
    LOCATIONS.region_dim                 AS region           COMMENT = 'Region',
    SALES_REP_ACTIVITY.territory_dim     AS territory        COMMENT = 'Sales Territory',
    SALES_REP_ACTIVITY.rep_name_dim      AS rep_name         COMMENT = 'Rep Name',
    CALL_DETAIL_RECORDS.phone_system_dim AS phone_system     COMMENT = 'Phone System',
    CALL_DETAIL_RECORDS.call_type_dim    AS call_type        COMMENT = 'Call Type',
    CALL_DETAIL_RECORDS.call_date_dim    AS call_date        COMMENT = 'Call Date'
  )
  METRICS (
    CLAIMS_SUBMISSIONS.denial_rate AS
      COUNT(CASE WHEN claim_status = 'DENIED' THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Percentage of claims denied (target: <5%)',
    CLAIMS_SUBMISSIONS.clean_claim_rate AS
      COUNT(CASE WHEN claim_status = 'PAID' AND paid_amount > 0 THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Percentage of claims paid on first submission (target: >95%)',
    CLAIMS_SUBMISSIONS.days_in_ar AS
      AVG(DATEDIFF(day, submission_date, adjudication_date))
      COMMENT = 'Average days from submission to adjudication',
    CLAIMS_DENIALS.total_denied_amount AS SUM(denied_amount)
      COMMENT = 'Total denied revenue in dollars',
    CALL_DETAIL_RECORDS.abandonment_rate AS
      COUNT(CASE WHEN abandoned THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Call abandonment rate (target: <5%)',
    CALL_DETAIL_RECORDS.fcr_rate AS
      COUNT(CASE WHEN first_call_resolution THEN 1 END) * 100.0 / NULLIF(COUNT(CASE WHEN NOT abandoned THEN 1 END), 0)
      COMMENT = 'First call resolution rate (target: >72%)',
    SALES_REP_ACTIVITY.conversion_rate AS
      COUNT(CASE WHEN referral_generated THEN 1 END) * 100.0 / NULLIF(COUNT(*), 0)
      COMMENT = 'Sales activity to referral conversion rate (target: 12-18%)'
  )
  COMMENT = 'Semantic view for Home Health DME operations: denials, sales, and call center'
  AI_VERIFIED_QUERIES (
    overall_denial_rate AS (
      QUESTION 'What is the overall denial rate?'
      SQL 'SELECT ROUND(COUNT(CASE WHEN claim_status = $$DENIED$$ THEN 1 END)*100.0/COUNT(*),2) AS denial_rate FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS'
    ),
    denial_rate_by_payer AS (
      QUESTION 'Show denial rate by payer'
      SQL 'SELECT payer_name, COUNT(*) AS total, ROUND(COUNT(CASE WHEN claim_status=$$DENIED$$ THEN 1 END)*100.0/COUNT(*),2) AS denial_rate_pct FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_SUBMISSIONS GROUP BY payer_name ORDER BY denial_rate_pct DESC'
    ),
    top_denial_reasons AS (
      QUESTION 'What are the top denial reasons?'
      SQL 'SELECT denial_code, denial_reason, root_cause, COUNT(*) AS cnt, SUM(denied_amount) AS total FROM HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS GROUP BY denial_code, denial_reason, root_cause ORDER BY cnt DESC LIMIT 10'
    ),
    top_reps AS (
      QUESTION 'Which sales reps generated the most referrals?'
      SQL 'SELECT rep_name, territory, COUNT(CASE WHEN referral_generated THEN 1 END) AS referrals, ROUND(COUNT(CASE WHEN referral_generated THEN 1 END)*100.0/COUNT(*),2) AS conv_rate FROM HOME_HEALTH_DEMO.RAW_DATA.SALES_REP_ACTIVITY GROUP BY rep_name, territory ORDER BY referrals DESC LIMIT 10'
    ),
    call_abandonment AS (
      QUESTION 'What is our call abandonment rate by phone system?'
      SQL 'SELECT phone_system, COUNT(*) AS total, ROUND(COUNT(CASE WHEN abandoned THEN 1 END)*100.0/COUNT(*),2) AS abandon_rate FROM HOME_HEALTH_DEMO.RAW_DATA.CALL_DETAIL_RECORDS GROUP BY phone_system'
    )
  );

GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE BILLING_ADMIN;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE SALES_MANAGER;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE ANALYST;
GRANT USAGE ON SEMANTIC VIEW SV_HOME_HEALTH_OPERATIONS TO ROLE EXECUTIVE;

SELECT 'Semantic View ready: HOME_HEALTH_DEMO.ANALYTICS.SV_HOME_HEALTH_OPERATIONS' AS status;

## Section 8: Cortex Search — AI Document Intelligence

### What Cortex Search Does
Traditional SQL cannot answer questions like "What does our CMN policy say about CPAP documentation requirements?" — that requires understanding **meaning**, not matching keywords.

**Cortex Search** provides hybrid search (semantic vector + keyword) over unstructured text:
- **No ML infrastructure** to build — Snowflake handles embedding and indexing automatically
- **Automatic refresh** when underlying documents change
- **Direct RAG integration** — feeds the Intelligence Agent in Section 9

### What We're Searching

| Service | Documents |
|---------|-----------|
| `HOME_HEALTH_POLICY_SEARCH` | 10 operational policy docs: claims submission, denial appeals, Medicare CMN, call center SLAs, payer billing rules, HIPAA compliance, sales territory policy, quality metrics, equipment authorization, referral management |

These documents power the AI Agent's ability to answer questions like "What is the timely filing limit for Medicare?" with context drawn from actual Home Health operational policies.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE SCHEMA DOCUMENTS;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

CREATE OR REPLACE TABLE HOME_HEALTH_POLICY_DOCUMENTS (
    document_id VARCHAR(20), document_title VARCHAR(500), policy_number VARCHAR(20),
    department VARCHAR(100), document_type VARCHAR(50), effective_date DATE,
    chunk_id INT, chunk_content TEXT);

CREATE OR REPLACE TABLE HOME_HEALTH_RAW_DOCUMENTS (
    filename VARCHAR(500), file_content VARCHAR(16777216));

-- Load markdown documents from stage
COPY INTO HOME_HEALTH_RAW_DOCUMENTS (filename, file_content)
FROM (
    SELECT METADATA$FILENAME, TO_VARCHAR($1)
    FROM @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DOCUMENTS_STAGE/
)
FILE_FORMAT = (TYPE='CSV' FIELD_DELIMITER=NONE RECORD_DELIMITER=NONE ESCAPE_UNENCLOSED_FIELD=NONE)
FORCE = TRUE;

SELECT filename, LENGTH(file_content) AS doc_length FROM HOME_HEALTH_RAW_DOCUMENTS ORDER BY filename;

In [ ]:
-- Chunk documents using SPLIT_TEXT_RECURSIVE_CHARACTER (markdown-aware, ~1500 chars, 200 overlap)
INSERT INTO HOME_HEALTH_POLICY_DOCUMENTS
WITH doc_metadata AS (
    SELECT
        filename,
        file_content,
        CASE WHEN filename LIKE '%01_claims%'    THEN 'CLM-001'
             WHEN filename LIKE '%02_denial%'    THEN 'DEN-001'
             WHEN filename LIKE '%03_medicare%'  THEN 'CMN-001'
             WHEN filename LIKE '%04_equipment%' THEN 'EQP-001'
             WHEN filename LIKE '%05_call%'      THEN 'CC-001'
             WHEN filename LIKE '%06_sales%'     THEN 'SLS-001'
             WHEN filename LIKE '%07_hipaa%'     THEN 'HIP-001'
             WHEN filename LIKE '%08_payer%'     THEN 'PAY-001'
             WHEN filename LIKE '%09_quality%'   THEN 'QM-001'
             WHEN filename LIKE '%10_referral%'  THEN 'REF-001' END AS policy_number,
        CASE WHEN filename LIKE '%01_claims%'    THEN 'Claims Submission Guidelines'
             WHEN filename LIKE '%02_denial%'    THEN 'Denial Appeal Procedures'
             WHEN filename LIKE '%03_medicare%'  THEN 'Medicare CMN Requirements'
             WHEN filename LIKE '%04_equipment%' THEN 'Equipment Authorization Policy'
             WHEN filename LIKE '%05_call%'      THEN 'Call Center SLA Standards'
             WHEN filename LIKE '%06_sales%'     THEN 'Sales Territory Policy'
             WHEN filename LIKE '%07_hipaa%'     THEN 'HIPAA Compliance for DME'
             WHEN filename LIKE '%08_payer%'     THEN 'Payer-Specific Billing Rules'
             WHEN filename LIKE '%09_quality%'   THEN 'Quality Metrics Definitions'
             WHEN filename LIKE '%10_referral%'  THEN 'Referral Management Guidelines' END AS document_title,
        CASE WHEN filename LIKE '%01_claims%'    THEN 'Revenue Cycle Management'
             WHEN filename LIKE '%02_denial%'    THEN 'Revenue Cycle - Appeals'
             WHEN filename LIKE '%03_medicare%'  THEN 'Clinical Documentation'
             WHEN filename LIKE '%04_equipment%' THEN 'Operations'
             WHEN filename LIKE '%05_call%'      THEN 'Contact Center Operations'
             WHEN filename LIKE '%06_sales%'     THEN 'Sales'
             WHEN filename LIKE '%07_hipaa%'     THEN 'Compliance'
             WHEN filename LIKE '%08_payer%'     THEN 'Revenue Cycle'
             WHEN filename LIKE '%09_quality%'   THEN 'Quality Improvement'
             WHEN filename LIKE '%10_referral%'  THEN 'Sales / Intake' END AS department,
        CASE WHEN filename LIKE '%guideline%' OR filename LIKE '%10_referral%' THEN 'Guideline'
             WHEN filename LIKE '%09_quality%' OR filename LIKE '%08_payer%'   THEN 'Reference'
             ELSE 'Policy' END AS document_type
    FROM HOME_HEALTH_RAW_DOCUMENTS
)
SELECT
    policy_number || '-' || ROW_NUMBER() OVER (PARTITION BY policy_number ORDER BY c.INDEX) AS document_id,
    document_title,
    policy_number,
    department,
    document_type,
    '2026-01-01'::DATE AS effective_date,
    ROW_NUMBER() OVER (PARTITION BY policy_number ORDER BY c.INDEX) AS chunk_id,
    c.VALUE::TEXT AS chunk_content
FROM doc_metadata,
LATERAL FLATTEN(
    SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(file_content, 'markdown', 1500, 200)
) c;

CREATE OR REPLACE CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH
    ON chunk_content
    WAREHOUSE = HOME_HEALTH_ANALYTICS_WH
    TARGET_LAG = '1 hour'
    AS (SELECT document_id, document_title, policy_number, department,
               document_type, effective_date, chunk_id, chunk_content
        FROM HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_DOCUMENTS);

GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE BILLING_ADMIN;
GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE SALES_MANAGER;
GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON CORTEX SEARCH SERVICE HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH TO ROLE ANALYST;

SELECT 'Cortex Search Service ready: HOME_HEALTH_POLICY_SEARCH' AS status;

## Section 9: Snowflake Intelligence — AI Agent

### What Makes This Different From a Chatbot
A regular chatbot only answers from training data. Snowflake Intelligence is an **agentic AI** that reasons, selects tools, executes queries, and synthesizes answers from multiple sources.

For a question like "What is our denial rate vs the policy target?" the agent:
1. Uses **Cortex Analyst** to query the claims data → actual denial rate from structured tables
2. Uses **Cortex Search** to retrieve the 5% target from the QM-001 quality metrics document
3. **Combines both** into one answer: data + policy context + recommended action

### Your Agent Has Two Tools

| Tool | What It Queries |
|------|----------------|
| `home_health_data` | Semantic View — claims, denials, sales, call records across all 3 use cases |
| `policy_search` | 10 policy documents — CMN requirements, SLA targets, billing rules, KPI definitions |

### Sample Questions to Try

**Data queries (Cortex Analyst)**
- What is our overall denial rate for Q1 2026?
- Show denial rates by payer, sorted highest to lowest
- Which locations have the highest denial rates?
- What is our call abandonment rate by phone system?
- Which sales reps generated the most referrals?

**Policy questions (Cortex Search)**
- What are the CMN requirements for oxygen equipment?
- What is the timely filing limit for Medicare claims?
- What documentation is needed to appeal a CO-16 denial?
- What are the SLA targets for the billing queue?

**Combined (both tools)**
- What is our denial rate vs the 5% policy target?
- Which payers have high denial rates and what are their billing rules?
- What is our abandonment rate vs the SLA standard?

This agent works in:
- **Snowflake Intelligence** web interface (zero code, share with anyone in the account)
- **Streamlit dashboard** (Section 11) via REST API
- **Any application** via the Snowflake Agents REST API

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS SNOWFLAKE_INTELLIGENCE;
GRANT USAGE ON DATABASE SNOWFLAKE_INTELLIGENCE TO ROLE PUBLIC;
CREATE SCHEMA IF NOT EXISTS SNOWFLAKE_INTELLIGENCE.AGENTS;
GRANT USAGE ON SCHEMA SNOWFLAKE_INTELLIGENCE.AGENTS TO ROLE PUBLIC;
GRANT CREATE AGENT ON SCHEMA SNOWFLAKE_INTELLIGENCE.AGENTS TO ROLE ACCOUNTADMIN;

CREATE OR REPLACE AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT
    COMMENT = 'Home Health DME Operations Agent: Cortex Analyst + Cortex Search'
    PROFILE = '{"display_name": "Home Health Operations Assistant", "color": "#0066CC"}'
    FROM SPECIFICATION $$
    {
        "models": {"orchestration": "claude-4-sonnet"},
        "instructions": {
            "orchestration": "You are Home Health DME operations assistant. Use home_health_data to query claims, denials, sales, and call center data. Use policy_search to find DME policies, billing rules, and SLA targets. Combine both for data-vs-target questions.",
            "response": "Be direct and actionable. Format as currency/percentages. Reference specific policy numbers. For denials, always mention root cause and next action.",
            "system": "You assist billing, sales, and call center teams at a large DME home health provider. Never expose individual patient identifiers."
        },
        "tools": [
            {
                "tool_spec": {
                    "type": "cortex_analyst_text_to_sql",
                    "name": "home_health_data",
                    "description": "Query DME operational data: claims, denials, appeals, sales activities, physician referrals, call records, agent performance."
                }
            },
            {
                "tool_spec": {
                    "type": "cortex_search",
                    "name": "policy_search",
                    "description": "Search Home Health DME policy documents: claims submission, denial appeal procedures, Medicare CMN requirements, call center SLAs, payer billing rules, HIPAA compliance."
                }
            }
        ],
        "tool_resources": {
            "home_health_data": {
                "semantic_view": "HOME_HEALTH_DEMO.ANALYTICS.SV_HOME_HEALTH_OPERATIONS",
                "execution_environment": {"type": "warehouse", "warehouse": "HOME_HEALTH_ANALYTICS_WH"},
                "query_timeout": 120
            },
            "policy_search": {
                "search_service": "HOME_HEALTH_DEMO.DOCUMENTS.HOME_HEALTH_POLICY_SEARCH",
                "max_results": 10,
                "columns": ["chunk_content", "document_title", "policy_number", "department"]
            }
        },
        "sample_prompts": [
            "What is our overall denial rate for Q1 2026?",
            "Show me denial rates by payer, sorted highest to lowest",
            "What is our denial rate vs the 5% policy target?",
            "What are the CMN requirements for oxygen equipment?",
            "Which locations have the highest denial rates?",
            "What is our call abandonment rate by phone system?",
            "What documentation is needed to appeal a CO-16 denial?",
            "What is the SLA target for the billing queue?",
            "Which sales reps generated the most referrals?",
            "What is the timely filing limit for Medicare claims?"
        ]
    }
    $$;

GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE PUBLIC;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE BILLING_ADMIN;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE SALES_MANAGER;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE ANALYST;
GRANT USAGE ON AGENT SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT TO ROLE EXECUTIVE;

SHOW AGENTS IN SCHEMA SNOWFLAKE_INTELLIGENCE.AGENTS;
SELECT 'Agent ready: SNOWFLAKE_INTELLIGENCE.AGENTS.HOME_HEALTH_OPERATIONS_AGENT' AS status;
SELECT 'Navigate to: AI & ML > Snowflake Intelligence in Snowsight' AS instructions;

## Section 10: FHIR R4 Pipeline — EHR Data Integration

### Phase 2 Use Case
Home Health currently receives clinical referrals from Epic and Cerner EHRs as fax orders. With FHIR R4, the referring hospital sends a structured JSON bundle directly — diagnoses, observations, medication orders, and encounter history all in one payload.

Snowflake's `VARIANT` type ingests the entire FHIR bundle without defining a schema upfront. `LATERAL FLATTEN` then queries any resource type on demand — without writing a parsing pipeline.

```
Denied claim (claim_id = CLM-001234, patient_id = PAT-0047821)
    ↓ JOIN on patient_id
FHIR Patient resource: John Smith, DOB 1948-03-12, male
    ↓ JOIN on fhir_patient_id
FHIR Condition: J44.1 - Severe COPD (SNOMED + ICD-10)
FHIR Observation: SpO2 = 86% (LOINC 59408-5)
FHIR MedicationRequest: E0431 - Portable oxygen concentrator
```

All 50 FHIR patients are drawn from `CLAIMS_DENIALS` — so the cross-use-case join is guaranteed to return results.

> **HIPAA Note:** All FHIR data is 100% synthetic. In production, Snowflake Business Critical includes the BAA, encryption at rest and in transit, private connectivity, and all the governance features from Section 4.

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE HOME_HEALTH_DEMO;
USE WAREHOUSE HOME_HEALTH_ANALYTICS_WH;

-- Raw FHIR table — entire bundle stored as VARIANT, no schema required
CREATE OR REPLACE TABLE HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES (
    bundle_id   VARCHAR,
    source_file VARCHAR,
    loaded_at   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    bundle_data VARIANT
);

-- PATTERN = '.*\.json' ensures only the JSON file loads, not the CSVs on the same stage
COPY INTO HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES (source_file, bundle_data)
FROM (
    SELECT METADATA$FILENAME, PARSE_JSON($1)
    FROM @HOME_HEALTH_DEMO.RAW_DATA.HOME_HEALTH_DATA_STAGE
)
FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = FALSE)
PATTERN = '.*\.json'
FORCE = TRUE;

-- Verify: show all resource types loaded from the bundle
SELECT res.value:resource:resourceType::VARCHAR AS resource_type, COUNT(*) AS count
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
GROUP BY resource_type ORDER BY count DESC;
-- Expected: Patient(50), Condition(100), Observation(150), MedicationRequest(100), Encounter(100)

In [ ]:
-- LATERAL FLATTEN — query nested FHIR JSON without any parsing code
CREATE OR REPLACE VIEW HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_PATIENT AS
SELECT
    res.value:resource:id::VARCHAR                  AS fhir_patient_id,
    res.value:resource:identifier[0]:value::VARCHAR  AS home_health_patient_id,
    res.value:resource:name[0]:family::VARCHAR        AS family_name,
    res.value:resource:name[0]:given[0]::VARCHAR      AS given_name,
    res.value:resource:gender::VARCHAR                AS gender,
    res.value:resource:birthDate::DATE                AS birth_date,
    DATEDIFF(year, res.value:resource:birthDate::DATE, CURRENT_DATE()) AS age_years,
    res.value:resource:address[0]:state::VARCHAR      AS state
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Patient';

CREATE OR REPLACE VIEW HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_CONDITION AS
SELECT
    res.value:resource:id::VARCHAR                                     AS condition_id,
    SPLIT_PART(res.value:resource:subject:reference::VARCHAR, '/', -1) AS fhir_patient_id,
    res.value:resource:code:coding[1]:code::VARCHAR                    AS icd10_code,
    res.value:resource:code:coding[1]:display::VARCHAR                 AS icd10_display,
    res.value:resource:code:text::VARCHAR                              AS condition_text,
    res.value:resource:onsetDateTime::TIMESTAMP                        AS onset_datetime
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Condition';

CREATE OR REPLACE VIEW HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_OBSERVATION AS
SELECT
    res.value:resource:id::VARCHAR                                     AS observation_id,
    SPLIT_PART(res.value:resource:subject:reference::VARCHAR, '/', -1) AS fhir_patient_id,
    res.value:resource:code:coding[0]:code::VARCHAR                    AS loinc_code,
    res.value:resource:code:coding[0]:display::VARCHAR                 AS observation_name,
    res.value:resource:valueQuantity:value::FLOAT                      AS value,
    res.value:resource:valueQuantity:unit::VARCHAR                     AS unit,
    res.value:resource:effectiveDateTime::DATE                         AS effective_date
FROM HOME_HEALTH_DEMO.FHIR_RAW.FHIR_BUNDLES b,
     LATERAL FLATTEN(input => b.bundle_data:entry) res
WHERE res.value:resource:resourceType::VARCHAR = 'Observation';

-- Cross-use-case view: clinical context of patients with denied claims
CREATE OR REPLACE VIEW HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_DENIED_PATIENTS_CLINICAL_PROFILE AS
SELECT p.home_health_patient_id, p.given_name, p.family_name, p.age_years, p.gender, p.state,
       cond.condition_text AS primary_diagnosis, cond.icd10_code,
       obs.value AS latest_spo2, obs.unit AS spo2_unit,
       d.denial_code, d.denial_reason, d.root_cause AS denial_root_cause,
       d.denied_amount, d.status AS denial_status
FROM HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_PATIENT p
JOIN HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_CONDITION cond ON cond.fhir_patient_id = p.fhir_patient_id
JOIN HOME_HEALTH_DEMO.RAW_DATA.CLAIMS_DENIALS d        ON d.patient_id = p.home_health_patient_id
LEFT JOIN HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_FHIR_OBSERVATION obs
    ON obs.fhir_patient_id = p.fhir_patient_id
    AND obs.loinc_code IN ('59408-5', '2708-6')
QUALIFY ROW_NUMBER() OVER (PARTITION BY p.home_health_patient_id ORDER BY obs.effective_date DESC NULLS LAST) = 1;

-- Query: clinical diagnoses of patients with denied claims
SELECT denial_root_cause, icd10_code, condition_text,
       COUNT(*) AS patients, SUM(denied_amount) AS denied_revenue
FROM HOME_HEALTH_DEMO.FHIR_ANALYTICS.V_DENIED_PATIENTS_CLINICAL_PROFILE
GROUP BY denial_root_cause, icd10_code, condition_text
ORDER BY denied_revenue DESC;

## Section 11: Streamlit Dashboard — 4-Tab Analytics App

### What We're Deploying
A fully interactive **Streamlit in Snowflake** app with:

| Tab | Content |
|-----|---------| 
| **Executive Summary** | Cross-domain KPIs — revenue at risk, denial rate, call center SLA, market penetration |
| **Denials Command Center** | Weekly trend, payer breakdown, root cause Pareto, appeal recovery funnel |
| **Sales Intelligence** | Market penetration map, rep leaderboard, physician referral pipeline, territory coverage gaps |
| **Call Center Operations** | Unified Avaya/Five9/RingCentral metrics, agent rankings, SLA heatmap |
| **AI Assistant** | Live chat connected to the Intelligence Agent with example questions |

### Deployment Steps
1. Navigate to **Snowsight → Projects → Streamlit → + Streamlit App**
2. Set database: `HOME_HEALTH_DEMO`, schema: `ANALYTICS`, warehouse: `HOME_HEALTH_ANALYTICS_WH`
3. Paste code from `home_health_analytics_app_sis.py`, add package `plotly`, click **Run**

> **No servers. No containers. No infrastructure.** Streamlit in Snowflake runs entirely inside your account, with the same governed data access as any SQL query.

In [ ]:
-- Grant dashboard access to all roles
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE BILLING_ADMIN;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE SALES_MANAGER;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE CALL_CENTER_LEAD;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE ANALYST;
GRANT USAGE ON STREAMLIT HOME_HEALTH_DEMO.ANALYTICS.HOME_HEALTH_OPERATIONS_DASHBOARD TO ROLE EXECUTIVE;

SELECT 'Home Health SnowDay HOL Complete!' AS status;

## Congratulations — You Built a Full Snowflake DME Analytics Platform!

### Snowflake Features Demonstrated

| Feature | Section | Business Value |
|---------|---------|----------------|
| Multi-cluster warehouses, auto-suspend | 1 | Zero cost when idle — vs Fabric's continuous F-SKU |
| Dynamic Data Masking + PHI classification | 4 | HIPAA compliance without data copies |
| Dynamic Tables (Bronze/Silver/Gold/Alerts) | 5 | No-orchestration pipeline — vs Fabric Data Factory |
| Snowflake Marketplace (CMS Medicare data) | 6 | Zero-ETL market intelligence |
| Semantic View + Cortex Analyst | 7 | NL queries for non-technical billing and sales teams |
| Cortex Search (10 policy documents) | 8 | RAG — 1 DDL vs Azure AI Search + OpenAI + custom code |
| Snowflake Intelligence Agent | 9 | Agentic AI over structured + unstructured data combined |
| FHIR R4 Pipeline (VARIANT + LATERAL FLATTEN) | 10 | EHR integration without ETL pipelines |
| Streamlit in Snowflake | 11 | Governed dashboards with embedded AI — no external hosting |

### Cleanup
To remove all objects: `DROP DATABASE IF EXISTS HOME_HEALTH_DEMO; DROP DATABASE IF EXISTS SNOWFLAKE_INTELLIGENCE;` then drop the three warehouses and six roles created in Section 1.